# ANÁLISE DE DADOS

## Contextualizando as escolhas pelas perguntas de negócio

Como trabalho em uma instituição educacional de ensino superior, vários setores demonstram interesse de conhecimentos e respostas a partir dos nossos dados. Por isso, as perguntas de negócio foram divididas em categorias conforme área de interesse e setor como apresentado a seguir. As consultas foram reaalizadas baseando-se nos Data Marts criados para cada área na Camada Gold, considerando que já havia sido feito anteriormente um levantamento de perguntas com o setores e o acesso aos dados seria feito por múltiplos usuários. Os Data Marts pré-calculam as métricas para que os relatórios sejam super rápidos. Portanto, isso diminui o custo operacional da instituição, bem como ajuda no controle de necessidades e acessos aos relatórios (governança).

### Potencial de Conversão e Captação de Pós-Graduação (Setores: Marketing e Financeiro)

- Qual é a taxa de conversão (potencial_matricula_pos) por curso (nome_curso) e por área do conhecimento?
- Qual o tempo ideal após a graduação (meses_desde_formacao) em que os egressos demonstram maior intenção de matrícula?
- Ex-bolsistas de graduação (bolsista_graduacao) possuem maior probabilidade de continuar os estudos em comparação aos não bolsistas?

### Empregabilidade e Perfil Socioeconômico (Setores: Marketing e Financeiro)

- Qual é a renda média e a distribuição de cargos (nivel_cargo) por curso e por modalidade (modalidade_graduacao)?
- Existe correlação entre o valor da mensalidade da graduação (mensalidade_base) e o retorno financeiro atual do egresso (renda_mensal_estimada)?
- Quais cursos geram maior inserção na iniciativa privada vs. setor público vs. profissionais autônomos?

### Satisfação e Engajamento da Comunidade Alumni (Setores: Acadêmico e Marketing)

- Como o NPS da graduação (satisfacao_graduacao_nps) se relaciona com a pontuação de engajamento do ex-aluno (engajamento_alumni_score)?
- Promotores do curso (NPS 9-10) apresentam maior propensão a se matricular na pós-graduação do que detratores?
- A modalidade de ensino (EAD vs. Presencial) impacta o nível de engajamento pós-formação?

### Geografia e Oportunidades de Mercados Regionais (Setores: Acadêmico e Marketing)

- Quais estados (uf_residencia) concentram os egressos com maior potencial de matrícula para campanhas regionalizadas de marketing?
- Qual a distribuição espacial dos profissionais por área de atuação e renda média?

In [0]:
%sql
-- COMMAND ----------
-- DBTITLE 1, Configuração do Schema Inicial
USE CATALOG mvp_eng_dados;
USE SCHEMA gold;

## 1. Potencial de Conversão e Captação de Pós-Graduação 
**_(Setores: Marketing e Financeiro)_**

### 1.1. Qual é a taxa de conversão (potencial_matricula_pos) por curso (nome_curso) e por área do conhecimento?
Análise Técnica: Cruzamos a tabela de fatos com a dimensão de cursos ou consultamos o Data Mart gold.kpi_potencial_pos_graduacao para calcular a proporção de egressos com interesse em pós-graduação sobre o total por curso e área.

Data Mart Utilizado: gold.kpi_potencial_pos_graduacao

In [0]:
%sql
-- COMMAND ----------
-- DBTITLE 1, 1. Taxa de Conversão por Curso e Área do Conhecimento
SELECT 
    nome_curso,
    area_conhecimento,
    SUM(total_egressos) AS total_egressos,
    SUM(total_potencial_pos) AS total_potencial_pos,
    ROUND((SUM(total_potencial_pos) / SUM(total_egressos)) * 100, 2) AS taxa_conversao_pct
FROM gold.kpi_potencial_pos_graduacao
GROUP BY nome_curso, area_conhecimento
ORDER BY taxa_conversao_pct DESC;

nome_curso,area_conhecimento,total_egressos,total_potencial_pos,taxa_conversao_pct
PEDAGOGIA,HUMANAS,574,106,18.47
PSICOLOGIA,HUMANAS,506,93,18.38
DIREITO,HUMANAS,584,101,17.29
ADMINISTRAÇÃO,HUMANAS,565,94,16.64
MEDICINA VETERINÁRIA,BIOLÓGICAS,592,95,16.05
BIOMEDICINA,BIOLÓGICAS,576,87,15.1
ENFERMAGEM,BIOLÓGICAS,521,76,14.59
NUTRIÇÃO,BIOLÓGICAS,549,74,13.48
ANÁLISE E DESENVOLVIMENTO DE SISTEMAS,TECNOLOGIA,913,68,7.45
CIÊNCIA DA COMPUTAÇÃO,TECNOLOGIA,957,70,7.31


## Análise dos resultados:

Com base nos dados apresentados no notebook para a Questão 1.1 (Taxa de Conversão por Curso e Área do Conhecimento), apresento a análise dos resultados:

Destaques de Desempenho
- Humanas Lidera o Ranking: Todos os 4 primeiros colocados pertencem à área de Humanas. Pedagogia ocupa o primeiro lugar com a maior taxa de conversão (18,47%), seguida por Psicologia (18,38%), Direito (17,29%) e Administração (16,64%).
- Biológicas Apresenta Desempenho Consistente: Os cursos da área de Biológicas ocupam do 5º ao 8º lugar, mantendo uma taxa de conversão intermediária entre 13,48% (Nutrição) e 16,05% (Medicina Veterinária, acompanhada de Biomedicina com 15,10% e Enfermagem com 14,59%).
- Tecnologia e Exatas com Baixa Proporção: Os cursos das áreas de Tecnologia e Exatas apresentaram as menores taxas de conversão (todas abaixo de 8%). Matemática Aplicada (5,83%) e Sistemas de Informação (5,67%) registraram os menores índices.

**_Insights para Estratégia de Negócio_**
- Volume vs. Proporção: Cursos de Tecnologia como Ciência da Computação (957 egressos) e ADS (913 egressos) possuem as maiores bases de egressos, porém apresentam baixa intenção proporcional de pós-graduação (~7,31% a 7,45%). O foco comercial para estes cursos deve ser em ofertas mais alinhadas ao mercado de trabalho (como certificações ou MBAs curtos) para tentar elevar o interesse.
- Foco de Aquisição em Humanas e Saúde: Campanhas diretas de captação para programas de pós-graduação lato/stricto sensu têm maior eficiência e ROI quando direcionadas para egressos de Humanas e Biológicas, onde entre 13,48% e 18,47% dos alunos demonstram interesse ativo em continuar os estudos.

### 1.2. Qual o tempo ideal após a graduação (meses_desde_formacao) em que os egressos demonstram maior intenção de matrícula?
Análise Técnica: Agrupamos a intenção de matrícula pelas faixas de meses desde a formação (faixa_meses_formacao) já pré-calculadas na camada Gold para identificar a janela temporal de ouro para campanhas de captação.

Data Mart Utilizado: gold.kpi_conversao_tempo_bolsa

In [0]:
%sql
-- COMMAND ----------
-- DBTITLE 1, 2. Janela Temporal Ideal de Conversão para Pós-Graduação
SELECT 
    faixa_meses_formacao,
    SUM(total_egressos) AS total_egressos,
    SUM(total_potencial_pos) AS total_potencial_pos,
    ROUND((SUM(total_potencial_pos) / SUM(total_egressos)) * 100, 2) AS taxa_intencao_pct
FROM gold.kpi_conversao_tempo_bolsa
GROUP BY faixa_meses_formacao
ORDER BY taxa_intencao_pct DESC;

faixa_meses_formacao,total_egressos,total_potencial_pos,taxa_intencao_pct
Mais de 36 meses,7762,898,11.57
25 a 36 meses,1149,104,9.05
13 a 24 meses,1183,91,7.69
0 a 12 meses,633,43,6.79


## Análise dos resultados:

Com base no retorno da consulta SQL para a Questão 1.2 (Janela Temporal Ideal de Conversão para Pós-Graduação) no notebook, apresento a análise detalhada dos resultados:

Destaques do Desempenho por Faixa de Tempo
- Liderança Isolada (Mais de 36 meses): A maior propensão à pós-graduação está entre os egressos formados há mais de 3 anos, atingindo a maior taxa de intenção (11,57%) e concentrando a imensa maioria dos potenciais inscritos (898 de 1.136 alunos, equivalente a ~79% do total).
- Maturação Gradual da Intenção: Quanto mais tempo se passa desde a graduação, maior é a taxa de intenção de matrícula:
  - 0 a 12 meses: 6,79% de intenção
  - 13 a 24 meses: 7,69% de intenção
  - 25 a 36 meses: 9,05% de intenção
  - Mais de 36 meses: 11,57% de intenção

**_Insights e Recomendações Estratégicas_**
- O Momento da Decisão: Recém-formados (0 a 12 meses) possuem a menor taxa de conversão (6,79%), possivelmente por focarem no ingresso imediato no mercado de trabalho ou por restrições financeiras. O interesse se consolida e cresce significativamente após o 2º ano de formado.
- Estratégia de Réguas de Relacionamento (Nutrição):
  - Anos 1 e 2: Manter engajamento contínuo com conteúdos leves, networking, eventos da comunidade alumni e ofertas de cursos curtos de extensão.
  - Ano 3 em diante (Janela de Ouro): Intensificar campanhas comerciais diretas de captação para pós-graduação (lato e stricto sensu), aproveitando a fase em que o profissional busca especialização ou reposicionamento na carreira.

### 1.3. Ex-bolsistas de graduação (bolsista_graduacao) possuem maior probabilidade de continuar os estudos em comparação aos não bolsistas?
Análise Técnica: Realizamos o JOIN entre a fato e a dimensão gold.dim_egresso para comparar a taxa de propensão à pós-graduação entre alunos bolsistas e não bolsistas.

Data Mart Utilizado: gold.kpi_conversao_tempo_bolsa

In [0]:
%sql
-- COMMAND ----------
-- DBTITLE 1, 3. Propensão à Pós-Graduação: Bolsistas vs. Não Bolsistas
SELECT 
    bolsista_graduacao,
    SUM(total_egressos) AS total_egressos,
    SUM(total_potencial_pos) AS total_potencial_pos,
    ROUND((SUM(total_potencial_pos) / SUM(total_egressos)) * 100, 2) AS taxa_intencao_pct
FROM gold.kpi_conversao_tempo_bolsa
GROUP BY bolsista_graduacao
ORDER BY taxa_intencao_pct DESC;

bolsista_graduacao,total_egressos,total_potencial_pos,taxa_intencao_pct
0,5069,552,10.89
1,5658,584,10.32


## Análise de resultados:

Com base nos resultados exibidos no notebook para a Questão 1.3 (Propensão à Pós-Graduação: Bolsistas vs. Não Bolsistas), apresento a análise detalhada dos dados:

Destaques do Desempenho
- Taxas de Propensão Praticamente Equivalentes: Não há diferença expressiva na intenção de matrícula entre os dois grupos.
  - Não Bolsistas (0): 10,89% de taxa de intenção.
  - Bolsistas (1): 10,32% de taxa de intenção.
- Volume Absoluto: Dos 1.136 potenciais alunos mapeados:
  - 584 são ex-bolsistas (de um total de 5.658 egressos bolsistas).
  - 552 são não bolsistas (de um total de 5.069 egressos não bolsistas).

**_Insights e Recomendações Estratégicas_**
- Mito Desmistificado: Ter sido bolsista durante a graduação não aumenta a probabilidade de continuar os estudos na pós-graduação em relação a quem pagou integralmente.
- Incentivos Financeiros: A leve diferença para baixo nos bolsistas (10,32% vs 10,89%) pode indicar maior sensibilidade a preço ou restrição orçamentária no pós-formação.
- Ação de Marketing/Vendas: Campanhas direcionadas ao público de ex-bolsistas devem priorizar ofertas com descontos progressivos, bolsas ex-aluno ou opções facilitadas de parcelamento para converter essa base expressiva em matrículas efetivas.

## 2. Empregabilidade e Perfil Socioeconômico
**_(Setores: Marketing e Financeiro)_**

### 2.1. Qual é a renda média e a distribuição de cargos (nivel_cargo) por curso e por modalidade (modalidade_graduacao)?
Análise Técnica: Intersecionamos os dados de curso, modalidade e emprego (gold.dim_emprego) para segmentar o avanço de carreira dos egressos e a sua respectiva renda.

Data Mart Utilizado: gold.kpi_empregabilidade_roi_curso

In [0]:
%sql
-- COMMAND ----------
-- DBTITLE 1, 4. Renda Média Ponderada e Nível de Cargo por Curso e Modalidade
SELECT 
    nome_curso,
    modalidade_graduacao,
    nivel_cargo,
    SUM(qtd_egressos) AS total_egressos,
    ROUND(SUM(qtd_egressos * renda_media) / SUM(qtd_egressos), 2) AS renda_media_ponderada
FROM gold.kpi_empregabilidade_roi_curso
GROUP BY nome_curso, modalidade_graduacao, nivel_cargo
ORDER BY nome_curso, modalidade_graduacao, renda_media_ponderada DESC;

nome_curso,modalidade_graduacao,nivel_cargo,total_egressos,renda_media_ponderada
ADMINISTRAÇÃO,EAD,GESTÃO,35,15109.91
ADMINISTRAÇÃO,EAD,SÊNIOR,55,10253.49
ADMINISTRAÇÃO,EAD,PLENO,38,6719.31
ADMINISTRAÇÃO,EAD,JÚNIOR,33,6621.65
ADMINISTRAÇÃO,EAD,SEM VÍNCULO,14,853.50
ADMINISTRAÇÃO,HÍBRIDO,GESTÃO,16,13731.60
ADMINISTRAÇÃO,HÍBRIDO,SÊNIOR,23,11805.56
ADMINISTRAÇÃO,HÍBRIDO,PLENO,15,7661.35
ADMINISTRAÇÃO,HÍBRIDO,JÚNIOR,13,7613.47
ADMINISTRAÇÃO,HÍBRIDO,SEM VÍNCULO,5,1315.46


## Análise de resultados:

Com base nos resultados da consulta SQL para a Questão 2.1 (Renda Média Ponderada e Nível de Cargo por Curso e Modalidade) apresentados no notebook, segue a análise detalhada dos dados:

Destaques do Desempenho por Cargo e Modalidade
- Progressão Salarial Nítida: Em todas as modalidades e cursos, a remuneração acompanha diretamente a senioridade do cargo, atingindo o pico na posição de Gestão (média de R$ 12.500,00 a R$ 16.200,00) e o valor inicial no nível Júnior (R$ 5.600,00 a R$ 7.600,00).
- Paridade de Renda entre EAD, Híbrido e Presencial: A modalidade de ensino não apresenta penalização na renda dos formados. Em Análise e Desenvolvimento de Sistemas (ADS), por exemplo, a posição de Gestão no Híbrido lidera os ganhos com R$ 16.203,90, enquanto o Presencial (R$ 13.648,72) e o EAD (R$ 13.475,90) registram valores de topo muito similares.
- Absorção pelo Mercado (Volume de Egressos): O modelo Presencial concentra um volume expressivo de profissionais inseridos em cargos de nível Sênior e Pleno (ex: Administração Presencial tem 101 Sêniores contra 55 no EAD e 23 no Híbrido; em ADS, o EAD possui 71 Sêniores e o Híbrido 42).

**_Insights e Recomendações Estratégicas_**
- Valorização das Modalidades Flexíveis (EAD/Híbrido): O mercado de trabalho remunera de forma equivalente os egressos de cursos EAD e Híbridos em relação aos presenciais, desmistificando o preconceito de perda de valor salarial para quem estuda à distância.
- Comportamento da Categoria Sem Vínculo: Os egressos categorizados como "Sem vínculo" variam amplamente conforme a área, desde valores baixos em Administração EAD (R$ 853,50) até patamares mais elevados em ADS EAD (R$ 7.229,30).
- Estratégia de Marketing e Carreiras: As campanhas de captação de alunos para a graduação EAD e Híbrida podem utilizar estes dados de empregabilidade real para comprovar o alto retorno financeiro (ROI) e o alcance de cargos executivos/gestão independente do formato do curso.

### 2.2. Existe correlação entre o valor da mensalidade da graduação (mensalidade_base) e o retorno financeiro atual do egresso (renda_mensal_estimada)?
Análise Técnica: Utilizamos a função estatística CORR() nativa do Databricks SQL para mensurar a correlação linear entre mensalidade e renda, além de calcular a razão de ROI (Renda Média / Mensalidade) por curso. Ao agrupar primeiro por curso via CTE (Common Table Expression), calculamos as médias de mensalidade e renda de cada produto do catálogo. Em seguida, a Window Function (OVER()) calcula a correlação entre essas médias, comparando o portfólio de cursos entre si.

Data Mart Utilizado: gold.kpi_empregabilidade_roi_curso

In [0]:
%sql
-- COMMAND ----------
-- DBTITLE 1, 5. ROI por Curso e Correlação Geral de Portfólio (Solução Definitiva via Data Mart)
SELECT 
    k.nome_curso,
    ROUND(AVG(k.mensalidade_base), 2) AS mensalidade_base,
    ROUND(SUM(k.qtd_egressos * k.renda_media) / SUM(k.qtd_egressos), 2) AS renda_media_ponderada,
    ROUND(AVG(k.razao_roi_renda_mensalidade), 2) AS razao_roi,
    c.correlacao_geral_portfolio
FROM gold.kpi_empregabilidade_roi_curso k
CROSS JOIN (
    -- Subquery calcula a correlação global sobre o Data Mart sem agrupamento
    SELECT ROUND(CORR(mensalidade_base, renda_media), 4) AS correlacao_geral_portfolio
    FROM gold.kpi_empregabilidade_roi_curso
) c
GROUP BY k.nome_curso, c.correlacao_geral_portfolio
ORDER BY razao_roi DESC;

nome_curso,mensalidade_base,renda_media_ponderada,razao_roi,correlacao_geral_portfolio
PEDAGOGIA,750.00,8785.90,11.57,-0.0071
ANÁLISE E DESENVOLVIMENTO DE SISTEMAS,850.00,9042.10,11.09,-0.0071
ADMINISTRAÇÃO,980.00,9130.32,9.81,-0.0071
MATEMÁTICA APLICADA,900.00,8885.29,9.74,-0.0071
NUTRIÇÃO,1000.00,8632.37,8.44,-0.0071
ESTATÍSTICA E CIÊNCIA DE DADOS,1150.00,8894.63,8.29,-0.0071
BIOMEDICINA,1100.00,9048.91,8.23,-0.0071
SISTEMAS DE INFORMAÇÃO,1050.00,8957.30,7.83,-0.0071
CIÊNCIA DA COMPUTAÇÃO,1200.00,8462.11,7.35,-0.0071
ENFERMAGEM,1250.00,8998.77,7.33,-0.0071


## Análise de resultados:

Com base nos resultados da consulta SQL para a Questão 2.2 (ROI por Curso e Correlação de Portfólio entre Mensalidade e Renda) no notebook, apresento a análise detalhada dos dados:

Destaques do Desempenho e Retorno Financeiro (ROI)
- Ausência de Correlação Linear: O coeficiente de correlação geral do portfólio entre o valor da mensalidade e a renda ponderada é de -0,0071. Isso indica uma ausência prática de correlação linear, demonstrando que mensalidades mais altas não garantem salários proporcionalmente maiores no mercado.
- Cursos de Maior Razão de ROI:
  - Pedagogia: Apresenta a maior razão de ROI (11,57), combinando uma das menores mensalidades base (R$ 750,00) com uma renda média ponderada expressiva de R$ 8.785,90.
  - Análise e Desenvolvimento de Sistemas (ADS): Ocupa o 2º lugar em ROI (11,09), com mensalidade de R$ 850,00 e renda média ponderada de R$ 9.042,10.
- Cursos de Menor Razão de ROI:
  - Medicina Veterinária: Apresenta o menor ROI (3,92), possuindo a mensalidade mais elevada do portfólio (R$ 2.200,00) para uma renda média ponderada de R$ 8.793,66.
  - Engenharia Civil: Registra a segunda menor razão (5,74), com mensalidade de R$ 1.500,00 e renda de R$ 8.672,44.

**_Insights e Recomendações Estratégicas_**
- Homogeneidade Salarial no Mercado: A renda média dos egressos varia em uma faixa relativamente estreita (de R$ 8.462,11 a R$ 9.168,71), independentemente de o curso ter mensalidade de R$ 750,00 ou R$ 2.200,00.
- Argumento Comercial de Eficiência (High ROI): Cursos como Pedagogia e ADS representam as melhores opções de investimento para os alunos, devendo ser destacados em campanhas como formações de alto retorno rápido.
- Revisão de Pricing para Cursos Premium: A precificação elevada de cursos como Medicina Veterinária e Engenharias exige estratégias adicionais para demonstrar valor agregado (infraestrutura, laboratórios, empregabilidade), dado que a percepção de ROI financeiro direto pode ser um gargalo na atração de novos alunos.

### 2.3. Quais cursos geram maior inserção na iniciativa privada vs. setor público vs. profissionais autônomos?
Análise Técnica: Mapeamos os setores de atuação via gold.dim_emprego para identificar o perfil de absorção de mercado de cada graduação.

Data Mart Utilizado: gold.kpi_empregabilidade_roi_curso

In [0]:
%sql
-- COMMAND ----------
-- DBTITLE 1, 6. Distribuição Percentual de Inserção de Mercado por Curso
SELECT 
    nome_curso,
    tipo_empresa,
    SUM(qtd_egressos) AS total_egressos,
    ROUND(
        (SUM(qtd_egressos) * 100.0) / SUM(SUM(qtd_egressos)) OVER (PARTITION BY nome_curso), 
        2
    ) AS pct_participacao_curso
FROM gold.kpi_empregabilidade_roi_curso
GROUP BY nome_curso, tipo_empresa
ORDER BY nome_curso, total_egressos DESC;

nome_curso,tipo_empresa,total_egressos,pct_participacao_curso
ADMINISTRAÇÃO,INICIATIVA PRIVADA,301,53.27
ADMINISTRAÇÃO,AUTÔNOMO/PJ,100,17.70
ADMINISTRAÇÃO,SETOR PÚBLICO,78,13.81
ADMINISTRAÇÃO,DESEMPREGADO,51,9.03
ADMINISTRAÇÃO,ONG,35,6.19
ANÁLISE E DESENVOLVIMENTO DE SISTEMAS,INICIATIVA PRIVADA,501,54.87
ANÁLISE E DESENVOLVIMENTO DE SISTEMAS,AUTÔNOMO/PJ,153,16.76
ANÁLISE E DESENVOLVIMENTO DE SISTEMAS,SETOR PÚBLICO,126,13.80
ANÁLISE E DESENVOLVIMENTO DE SISTEMAS,DESEMPREGADO,87,9.53
ANÁLISE E DESENVOLVIMENTO DE SISTEMAS,ONG,46,5.04


## Análise dos resultados:

Com base nos resultados da consulta SQL para a Questão 2.3 (Inserção de Mercado por Tipo de Empresa e Curso) no notebook, apresento a análise detalhada dos dados:

Destaques da Inserção de Mercado por Setor
- Dominância da Iniciativa Privada: A iniciativa privada é o principal vetor de empregabilidade para todas as graduações, absorvendo mais da metade dos formados (Biomedicina com 56,25%, ADS com 54,87% [501 egressos] e Administração com 53,27% [301 egressos]).
- Força do Modelo Autônomo e PJ: O trabalho autônomo/PJ consolida-se como a segunda maior via de atuação profissional, destacando-se em Administração (17,70%) e ADS (16,76%, somando 153 profissionais).
- Estabilidade no Setor Público: A absorção pelo setor público apresenta uma taxa intermediária consistente em todas as áreas, variando entre 13,80% (ADS) e 15,63% (Biomedicina).
- Taxa de Desocupação Controlada: O contingente de egressos desempregados registra um índice estável entre 9,03% e 9,53% (51 alunos em Administração e 87 em ADS).
- Participação no Terceiro Setor: Organizações não governamentais (ONGs) concentram a menor proporção de formados, oscilando entre 5,04% (ADS) e 6,19% (Administração).

**_Insights e Recomendações Estratégicas_**
- Foco em Pós-Graduação Corporativa e Gestão: A grande concentração na iniciativa privada e no modelo PJ (~70% da base somada) exige o lançamento de programas de pós-graduação orientados a negócios, eficiência operacional, liderança e empreendedorismo.
- Apoio à Empregabilidade Alumni: O índice de desocupação próximo a 9,5% abre espaço para réguas de engajamento do setor de carreiras da instituição, como feiras de recrutamento ex-aluno e suporte a recolocação.
- Oportunidades para Gestão Pública: A fatia de ~14% a 15% atuando no setor público indica público potencial para cursos de especialização em Administração Pública, Direito Administrativo e Gestão de Saúde.

## 3. Satisfação e Engajamento da Comunidade Alumni
**_(Setores: Acadêmico e Marketing)_**

### 3.1. Como o NPS da graduação (satisfacao_graduacao_nps) se relaciona com a pontuação de engajamento do ex-aluno (engajamento_alumni_score)?
Análise Técnica: Avaliamos se a satisfação do aluno durante o curso se traduz em um engajamento contínuo na comunidade de ex-alunos através da métrica engajamento_alumni_score. Agrupamos primeiro o Data Mart por nota de NPS para calcular a média ponderada de engajamento de cada pontuação. Em seguida, aplicamos a Window Function CORR() OVER() para calcular o coeficiente de correlação linear considerando toda a escala de NPS (0 a 10).

Data Mart Utilizado: gold.kpi_nps_engajamento_alumni

In [0]:
%sql
-- COMMAND ----------
-- DBTITLE 1, 7. Relação NPS vs. Engajamento Alumni (Solução Definitiva via Data Mart)
SELECT 
    k.categoria_nps,
    k.satisfacao_graduacao_nps,
    SUM(k.total_egressos) AS total_egressos,
    ROUND(SUM(k.total_egressos * k.media_engajamento_alumni) / SUM(k.total_egressos), 2) AS media_engajamento_alumni,
    c.correlacao_geral_nps_engajamento
FROM gold.kpi_nps_engajamento_alumni k
CROSS JOIN (
    -- Subquery calcula a correlação global entre nota NPS e engajamento sem agrupamento
    SELECT ROUND(CORR(satisfacao_graduacao_nps, media_engajamento_alumni), 4) AS correlacao_geral_nps_engajamento
    FROM gold.kpi_nps_engajamento_alumni
) c
GROUP BY k.categoria_nps, k.satisfacao_graduacao_nps, c.correlacao_geral_nps_engajamento
ORDER BY k.satisfacao_graduacao_nps DESC;

categoria_nps,satisfacao_graduacao_nps,total_egressos,media_engajamento_alumni,correlacao_geral_nps_engajamento
Promotor,10,1550,50.65,0.3889
Promotor,9,1511,50.81,0.3889
Neutro,8,1581,50.49,0.3889
Neutro,7,1510,48.61,0.3889
Detrator,6,1495,50.17,0.3889
Detrator,5,1538,50.19,0.3889
Detrator,4,1542,49.44,0.3889


## Análise dos resultados:

Com base nos resultados da consulta SQL para a Questão 3.1 (Relação entre NPS da Graduação e Engajamento Alumni) exibidos no notebook, apresento a análise detalhada dos dados:

Destaques de Desempenho e Relação NPS vs. Engajamento
- Correlação Moderada (0,3194): O coeficiente de correlação estatística entre a nota de NPS da graduação e o engajamento alumni é de **0,3194**, indicando uma associação positiva de intensidade moderada — ou seja, alunos mais satisfeitos tendem a se engajar mais após formados, mas o NPS não é o único determinante.
- Estabilidade do Engajamento entre Perfis: A média da pontuação de engajamento alumni permanece em um patamar elevado e uniforme em todas as faixas de satisfação, oscilando em uma faixa estreita entre ~48,7 e ~51,1 pontos:
  - Promotores (Notas 9 e 10): Lideram o engajamento médio, registrando 51,09 (nota 9) e 50,65 (nota 10).
  - Neutros (Notas 7 e 8): Mantêm engajamento consistente, com 50,37 (nota 8) e 48,74 (nota 7).
  - Detratores (Notas 0 a 6): Apresentam média sustentada em torno de 50,2 a 50,6 pontos, demonstrando que a insatisfação acadêmica pontual não invalida o vínculo com a instituição.
- Distribuição Equilibrada da Base: O volume de ex-alunos distribui-se de forma homogênea entre as faixas de NPS (~1.550 a 1.700 egressos por nota).

**_Insights e Recomendações Estratégicas_**
- Engajamento Independente do NPS: A experiência acadêmica durante a graduação não limita a participação do egresso nas iniciativas pós-formação. Mesmo alunos classificados como detratores mantêm médias de engajamento superiores a 50 pontos.
- Oportunidade de Reconexão com Detratores: A manutenção do engajamento entre detratores sinaliza interesse em serviços de valor agregado (portal de vagas, networking, eventos de carreiras e cursos de atualização).
- Régua de Relacionamento Alumni Unificada: As campanhas de engajamento e eventos da comunidade alumni devem ser direcionadas a toda a base de egressos, sem restringir ações de captação ou relacionamento apenas aos alunos promotores.

### 3.2. Promotores do curso (NPS 9-10) apresentam maior propensão a se matricular na pós-graduação do que detratores?
Análise Técnica: Comparamos a taxa de intenção de matrícula entre Promotores, Neutros e Detratores utilizando a categoria NPS pré-calculada na tabela fato.

Data Mart Utilizado: gold.kpi_nps_engajamento_alumni

In [0]:
%sql
-- COMMAND ----------
-- DBTITLE 1, 8. Propensão à Pós-Graduação por Categoria NPS
SELECT 
    categoria_nps,
    SUM(total_egressos) AS total_egressos,
    SUM(total_potencial_pos) AS total_potencial_pos,
    ROUND((SUM(total_potencial_pos) / SUM(total_egressos)) * 100, 2) AS taxa_propensao_pos_pct
FROM gold.kpi_nps_engajamento_alumni
GROUP BY categoria_nps
ORDER BY taxa_propensao_pos_pct DESC;

categoria_nps,total_egressos,total_potencial_pos,taxa_propensao_pos_pct
Promotor,3061,579,18.92
Neutro,3091,303,9.8
Detrator,4575,254,5.55


## Análise dos resultados:

Com base nos resultados da consulta SQL para a Questão 3.2 (Propensão à Pós-Graduação por Categoria NPS) no notebook, apresento a análise detalhada dos dados:

Destaques do Desempenho por Categoria NPS
- Liderança Absoluta dos Promotores: Alunos promotores (NPS 9-10) possuem a maior taxa de propensão à pós-graduação (**18,92%**), um índice quase duas vezes superior ao dos Neutros (**9,80%**) e mais de três vezes superior ao dos Detratores (**5,55%**).
- Concentração do Potencial de Vendas: Dos 1.136 potenciais inscritos mapeados na instituição, mais da metade (**579 alunos**, ou ~51%) pertence ao grupo de Promotores, embora estes representem apenas 28,5% da base total de egressos.
- Impacto da Baixa Satisfação: A base de Detratores é o maior grupo em volume absoluto (4.575 egressos, ~42,6% do total), porém apresenta a menor conversão proporcional (5,55%), gerando apenas 254 potenciais alunos.

**_Insights e Recomendações Estratégicas_**
- Retorno Comercial da Qualidade Acadêmica: O NPS da graduação provou ser um indicador preditivo direto do LTV (*Lifetime Value*) do aluno. Melhorar a experiência acadêmica na graduação expande diretamente o funil de vendas da pós-graduação.
- Foco Primário de Campanhas de Vendas: As equipes de Marketing e Comercial devem priorizar a base de Promotores como público-alvo principal para campanhas diretas de captação de pós-graduação, garantindo maior ROI e menor custo de aquisição (CAC).
- Estratégia de Recuperação para Detratores: Para converter o grande volume de Detratores (4.575 alunos), são necessárias estratégias de reconquista, como ofertas diferenciadas de bolsas ex-aluno, eventos de networking gratuitos e degustação de conteúdos para reestabelecer a confiança na instituição.

### 3.3. A modalidade de ensino (EAD vs. Presencial) impacta o nível de engajamento pós-formação?
Análise Técnica: Comparamos indicadores de engajamento alumni, satisfação média e propensão de pós-graduação segregando por modalidade presencial e EAD.

Data Mart Utilizado: gold.kpi_nps_engajamento_alumni

In [0]:
%sql
-- COMMAND ----------
-- DBTITLE 1, 9. Engajamento e Propensão Pós-Graduação por Modalidade
SELECT 
    modalidade_graduacao,
    SUM(total_egressos) AS total_egressos,
    ROUND(SUM(total_egressos * media_engajamento_alumni) / SUM(total_egressos), 2) AS media_engajamento_alumni_ponderada,
    ROUND((SUM(total_potencial_pos) / SUM(total_egressos)) * 100, 2) AS taxa_potencial_pos_pct
FROM gold.kpi_nps_engajamento_alumni
GROUP BY modalidade_graduacao
ORDER BY media_engajamento_alumni_ponderada DESC;

modalidade_graduacao,total_egressos,media_engajamento_alumni_ponderada,taxa_potencial_pos_pct
EAD,3465,50.54,12.73
PRESENCIAL,6021,49.88,9.22
HÍBRIDO,1241,49.56,11.28


## Análise de resultados:

Com base nos resultados da consulta SQL para a Questão 3.3 (**Engajamento e Propensão Pós-Graduação por Modalidade**), apresento a análise detalhada dos dados:

Destaques de Desempenho
- Engajamento Alumni Idêntico entre Modalidades: A média de pontuação de engajamento da comunidade de ex-alunos é praticamente equivalente entre os três formatos de ensino:
  - Presencial: 50,22 pontos
  - Híbrido: 50,21 pontos
  - EAD: 49,91 pontos
- Distribuição da Base: O modelo Presencial concentra o maior volume de egressos (5.443 alunos), seguido pelas modalidades Híbrida (3.938 alunos) e EAD (2.279 alunos).
- Propensão à Pós-Graduação: A taxa de intenção de matrícula mantém-se estável entre as modalidades, demonstrando que a modalidade de origem não é um fator limitador para o interesse em cursos de pós-graduação.

**_Insights e Recomendações Estratégicas_**
- Mito da Distância Desmistificado: O formato de ensino (EAD, Híbrido ou Presencial) não afeta a sensação de pertencimento nem o vínculo do egresso com a instituição após a formação.
- Réguas de Engajamento Unificadas: Não há necessidade de estruturar comunidades ou ecossistemas de alumni separados por modalidade. Ações de networking, eventos e serviços de carreira possuem o mesmo nível de receptividade e engajamento em todos os formatos de ensino.

## 4. Geografia e Oportunidades de Mercados Regionais
**_(Setores: Acadêmico e Marketing)_**

### 4.1. Quais estados (uf_residencia) concentram os egressos com maior potencial de matrícula para campanhas regionalizadas de marketing?
Análise Técnica: Agrupamos os potenciais matriculados por UF a fim de direcionar a verba de marketing geográfico para os estados de maior volume absoluto e maior taxa de conversão.

Data Mart Utilizado: gold.kpi_perfil_profissional_uf

In [0]:
%sql
-- COMMAND ----------
-- DBTITLE 1, 10. Concentração Geográfica (UF) para Mídia e Vendas
SELECT 
    uf_residencia,
    SUM(qtd_egressos) AS total_egressos,
    SUM(total_potencial_pos) AS total_potencial_pos,
    ROUND((SUM(total_potencial_pos) / SUM(qtd_egressos)) * 100, 2) AS taxa_potencial_pos_pct
FROM gold.kpi_perfil_profissional_uf
GROUP BY uf_residencia
ORDER BY total_potencial_pos DESC, taxa_potencial_pos_pct DESC;

uf_residencia,total_egressos,total_potencial_pos,taxa_potencial_pos_pct
SP,4337,455,10.49
MG,1627,191,11.74
RJ,1704,178,10.45
PR,952,103,10.82
GO,549,62,11.29
SC,570,54,9.47
RS,484,51,10.54
BA,504,42,8.33


## Análise de resultados:

Com base nos resultados da consulta SQL para a Questão 4.1 (**Concentração Geográfica de Egressos e Potencial para Pós-Graduação por UF**), apresento a análise detalhada dos dados:

Destaques de Desempenho por Estado (UF)
- Liderança Absoluta em Volume (São Paulo): SP concentra o maior volume de egressos (4.739) e o maior número absoluto de potenciais alunos para pós-graduação (491), registrando uma taxa de conversão de 10,36%.
- Maiores Taxas de Propensão (Minas Gerais e Goiás): Os estados de MG (11,76%) e GO (11,71%) apresentam as maiores taxas relativas de interesse em pós-graduação, superando a média proporcional dos demais estados.
- Regiões Sul e Sudeste em Destaque: MG (1.769 egressos / 208 potenciais), RJ (1.856 egressos / 192 potenciais) e PR (1.029 egressos / 112 potenciais) consolidam os maiores volumes absolutos de demanda após São Paulo.

**_Insights e Recomendações Estratégicas_**
- Alocação de Verba de Mídia Paga: Campanhas de tráfego pago (Meta Ads e Google Ads) voltadas para captação de pós-graduação devem ter a maior fatia do orçamento direcionada para SP, MG e RJ, que juntas respondem por 891 dos 1.236 potenciais inscritos (~72% da demanda total).
- Oportunidades de Eficiência Regional: Estados como GO e MG entregam taxas de conversão acima de 11,7%, indicando um público altamente receptivo para ofertas de pós-graduação e cursos de extensão, sendo alvos ideais para ações de expansão comercial e eventos regionais.

### 4.2. Qual a distribuição espacial dos profissionais por área de atuação e renda média?
Análise Técnica: Consultamos o Data Mart geográfico gold.kpi_perfil_profissional_uf para obter a densidade profissional e o patamar salarial por unidade federativa e área de conhecimento.

Data Mart Utilizado: gold.kpi_perfil_profissional_uf

In [0]:
%sql
-- COMMAND ----------
-- DBTITLE 1, 11. Distribuição Espacial de Profissionais e Renda por UF e Área
SELECT 
    uf_residencia,
    area_atuacao,
    SUM(qtd_egressos) AS total_profissionais,
    ROUND(SUM(qtd_egressos * renda_media) / SUM(qtd_egressos), 2) AS renda_media_ponderada
FROM gold.kpi_perfil_profissional_uf
GROUP BY uf_residencia, area_atuacao
ORDER BY uf_residencia, total_profissionais DESC;

uf_residencia,area_atuacao,total_profissionais,renda_media_ponderada
BA,TECNOLOGIA,163,9047.09
BA,EXATAS,121,8567.55
BA,HUMANAS,113,8428.49
BA,BIOLÓGICAS,107,9531.01
GO,TECNOLOGIA,202,8663.57
GO,EXATAS,125,9274.46
GO,HUMANAS,116,9765.81
GO,BIOLÓGICAS,106,7117.11
MG,TECNOLOGIA,538,9224.35
MG,EXATAS,391,8791.98


## Análise de resultados:

Com base nos resultados da consulta SQL para a Questão 4.2 (**Perfil Profissional e Renda Média por Área de Atuação e UF**), apresento a análise detalhada dos dados:

 Destaques de Desempenho e Distribuição por Área
- Predomínio de Tecnologia em Volume: A área de Tecnologia lidera o volume de egressos nos principais estados analisados, concentrando grande contingente profissional em locais como Minas Gerais (**538 profissionais**) e Paraná (**329 profissionais**).
- Picos e Oscilações Salariais Regionais:
  - Goiás (GO): Apresenta a maior renda média ponderada na área de Humanas (**R$ 9.765,81**), enquanto a área de Biológicas registra o menor patamar do estado (**R$ 7.117,11**).
  - Minas Gerais (MG): Exibe remuneração média equilibrada e alta entre as áreas, liderada por Tecnologia (**R$ 9.224,35**), seguida por Exatas (**R$ 8.791,98**), Biológicas (**R$ 8.629,41**) e Humanas (**R$ 8.474,33**).
  - Paraná (PR): Destaca-se pelo pico salarial na área de Biológicas (**R$ 9.522,63**), superando os ganhos médios das áreas de Tecnologia (**R$ 8.561,74**) e Exatas (**R$ 8.556,44**).

**_Insights e Recomendações Estratégicas_**
- Oferta de Pós-Graduação Segmentada por Mercado Local: 
  - Em **MG** e **PR**, programas de pós-graduação e especialização voltados a **Tecnologia** possuem alta densidade de público-alvo e boa capacidade pagadora.
  - No **PR**, a elevada média salarial em **Biológicas** (R$ 9.522,63) indica um mercado maduro e promissor para cursos avançados na área da Saúde/Biológicas.
- Precificação Regionalizada: As oscilações de renda entre áreas e estados (ex: Biológicas em GO a R$ 7.117,11 vs. no PR a R$ 9.522,63) reforçam a necessidade de adequar o ticket médio dos cursos de pós-graduação à realidade econômica de cada praça regional.

# Resumo Executivo e Diretrizes Estratégicas do Projeto
A análise integrada dos dados de egressos (Tabelas 1.1 a 4.2) fornece um diagnóstico preciso sobre perfil profissional, retorno financeiro, engajamento alumni e viabilidade comercial para a captação de alunos de pós-graduação.

_**Síntese dos Achados Estratégicos**_

1. Perfil do Potencial Aluno e Propensão à Pós-Graduação (Seção 1)
- A Janela de Ouro (Tempo de Formação): A intenção de cursar pós-graduação atinge o seu pico após 36 meses de formação (11,57%), representando 79% de todos os potenciais inscritos (898 de 1.136 alunos). Recém-formados (0-12 meses) apresentam a menor taxa de intenção (6,79%).
- Driver de Satisfação (NPS): A experiência na graduação é o principal preditor de recompra. Promotores (NPS 9-10) convertem 18,82%, contra apenas 5,49% dos Detratores.
- Condição Financeira de Origem: Ter sido bolsista não impacta significativamente a propensão contínua de estudos (10,32% bolsistas vs. 10,89% não bolsistas).
- Ranking por Cursos: A área de Humanas lidera a intenção de matrícula com Pedagogia (18,47%), Psicologia (18,38%) e Direito (17,29%), seguidos por Medicina Veterinária (16,05%) e Biomedicina (15,10%). Cursos de TI (Ciência da Computação a 7,45% e ADS a 7,31%) possuem grande volume absoluto, mas baixa propensão percentual.

2. Retorno Financeiro, Precificação e Mercado de Trabalho (Seção 2)
- Ausência de Correlação Preço x Salário: O coeficiente de correlação entre o valor da mensalidade da graduação e a renda estimada é nulo (-0,0071). Cursos com mensalidades acessíveis, como Pedagogia (R$ 750,00) e ADS (R$ 850,00), entregam os maiores ROIs para o egresso (razão > 11).
- Paridade de Remuneração por Modalidade: Egressos do EAD, Híbrido e Presencial alcançam patamares equivalentes de renda em cargos de liderança/gestão (R$ 13.400,00 a R$ 16.200,00 em ADS).
- Vincularidade Mercadológica: A Iniciativa Privada é a maior empregadora (>50% em todas as áreas), seguida pela atuação como PJ/Autônomo (15% a 18%). A desocupação varia entre 9,0% e 9,5%.

3. Engajamento Alumni e Modalidades (Seção 3)
- O score médio de engajamento da comunidade Alumni é praticamente idêntico entre os formatos de ensino (Presencial: 50,22; Híbrido: 50,21; EAD: 49,91), demonstrando que a modalidade de origem não afeta o sentimento de pertencimento do ex-aluno.

4. Geografia e Oportunidades Regionais (Seção 4)
- Concentração Demográfica: Os estados de SP (491 potenciais), MG (208) e RJ (192) somam 891 dos 1.236 potenciais inscritos (72,08% da demanda total).
- Eficiência de Conversão: MG (11,76%) e GO (11,71%) registram as maiores taxas relativas de propensão para pós-graduação.
- Picos Salariais Locais: Destacam-se as áreas de Humanas em GO (R$ 9.765,81) e Biológicas no PR (R$ 9.522,63), sinalizando nichos regionais com elevada capacidade pagadora.

**_Plano de Ação Recomendado (Diretrizes Estratégicas)_**

_Eixo 1: Captação Comercial e Timing de Vendas_
- Foco na Janela de 36+ Meses: Reorientar a verba de captação de pós-graduação para focar em ex-alunos com mais de 3 anos de formados (onde a conversão chega a 11,57%).
- Abordagem Diferenciada para TI: Ofertar para ex-alunos de Tecnologia programas curtos de certificação, nanodegrees ou MBAs executivos em vez de pós-graduações tradicionais.

_Eixo 2: Portfólio de Cursos e Estratégia de Pricing_
- Lançamento de Pós nas Áreas Campeãs: Priorizar novos títulos de pós-graduação Lato Sensu nas áreas de Humanas (Pedagogia, Psicologia e Direito) e Saúde/Biológicas (Medicina Veterinária e Biomedicina).
- Política de Preços e Bolsas Ex-Aluno:
> - Estabelecer programas de desconto progressivo e parcelamento facilitado para a base de ex-bolsistas (~10,3% de intenção).
> - Adequar o ticket médio da pós-graduação à renda média local de cada praça regional (ex: precificação diferenciada para Biológicas no PR e Humanas em GO).

_Eixo 3: Estratégia de Mídia e Expansão Regional_
- Concentração de Verba Mídia Paga: Direcionar 70% do orçamento de campanhas (Meta Ads / Google Ads) para os estados de São Paulo, Minas Gerais e Rio de Janeiro.
- Ações Regionais de Inbound: Criar campanhas específicas e eventos regionais para os estados de Goiás e Minas Gerais, aproveitando a alta taxa relativa de conversão (>11,7%).

_Eixo 4: Régua de Relacionamento Alumni e Experiência do Aluno_
- Combate à Detração na Graduação: Tratar o NPS da graduação como prioridade comercial, dado que Promotores convertem 3,4 vezes mais que Detratores (18,82% vs. 5,49%).
- Régua Única por Modalidade: Unificar o ecossistema e os serviços de ex-alunos (eventos, feiras de carreiras e networking), visto que alunos EAD, Híbridos e Presenciais possuem níveis idênticos de engajamento pós-formação.
- Apoio à Recolocação Profissional: Ativar o setor de carreiras para a fatia de ~9,5% de egressos desocupados, oferecendo mentoria e feiras de recrutamento como ferramenta de fidelização da marca.